# License Plate Recognition

- **Author**: Balázs Róna
- **Matriculation Number**: 12402137

In [ ]:
%pip install -U torch==2.8.0 torchvision==0.23.0
%pip install -U albumentations opencv-python scikit-image shapely pycocotools easyocr tqdm matplotlib pandas

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import json
import time
from pathlib import Path

import cv2
import numpy as np

import albumentations as A
from albumentations.pytorch import ToTensorV2

## Helper Functions

In [ ]:
def letterbox(img, mask, target_hw=(512, 512), border_value=(114, 114, 114)):
    """
    Letterbox/pad image to target size.
    
    @param img: The image to resize/pad.
    @param mask: The mask to resize/pad.
    @param target_hw: Target image (height, width) tuple.
    @param border_value: Padding border color.
    @return: (img_padded, mask_padded) - Tuple of padded image and mask.
    """
    
    height, width = img.shape[:2]
    target_height, target_width = target_hw
    scale = min(target_width / width, target_height / height)
    new_width, new_height = int(round(width * scale)), int(round(height * scale))
    img_resized = cv2.resize(img, (new_width, new_height), interpolation=cv2.INTER_LINEAR)
    mask_resized = cv2.resize(mask, (new_width, new_height), interpolation=cv2.INTER_NEAREST)

    # pad to target size
    top = (target_height - new_height) // 2
    bottom = target_height - new_height - top
    left = (target_width - new_width) // 2
    right = target_width - new_width - left

    img_padded = cv2.copyMakeBorder(img_resized, top, bottom, left, right,
                                    cv2.BORDER_CONSTANT, value=border_value)
    mask_padded = cv2.copyMakeBorder(mask_resized, top, bottom, left, right,
                                     cv2.BORDER_CONSTANT, value=0)
    return img_padded, mask_padded

def bbox_to_mask(height, width, bbox):
    """
    bbox = (x_min, y_min, x_max, y_max) in pixel coords
    """
    
    mask = np.zeros((height, width), dtype=np.uint8)
    x1, y1, x2, y2 = map(int, bbox)
    mask[y1:y2, x1:x2] = 1
    return mask

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

def get_train_transform(target_hw=(512, 512)):
    """
    Create random transformations to augment training images.
    
    @param target_hw: Target image (height, width) tuple.
    @return: Albumentations composition of random transformations.
    """
    return A.Compose([
        A.LongestMaxSize(max(target_hw), interpolation=cv2.INTER_LINEAR),
        A.PadIfNeeded(min_height=target_hw[0], min_width=target_hw[1],
                      border_mode=cv2.BORDER_CONSTANT, fill=(114, 114, 114), fill_mask=0),
        A.HorizontalFlip(p=0.5),
        A.Affine(scale=(0.9, 1.1), rotate=(-8, 8), shear=(-5, 5), translate_percent=(0.02, 0.02), p=0.5),
        A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.02, p=0.5),
        A.MotionBlur(blur_limit=3, p=0.1),
        A.GaussNoise(std_range=(0.01, 0.05), mean_range=(0.0, 0.0), p=0.15),
        A.ImageCompression(quality_range=(60, 100), p=0.2),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2()
    ])

def get_val_transform(target_hw=(512, 512)):
    """
    Create random transformations to augment validation images.
    
    @param target_hw: Target image (height, width) tuple.
    @return: Albumentations composition of random transformations.
    """
    return A.Compose([
        A.LongestMaxSize(max(target_hw), interpolation=cv2.INTER_LINEAR),
        A.PadIfNeeded(min_height=target_hw[0], min_width=target_hw[1],
                      border_mode=cv2.BORDER_CONSTANT, fill=(114, 114, 114), fill_mask=0),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2()
    ])

class PlateSegDataset(Dataset):
    def __init__(self, root, split_txt, transform):
        """
        Assumed directory structure:
        
        root/
          images/
          masks/
          splits/train.txt (file stems: without extension)
        
        @param root: Dataset root directory.
        @param split_txt: Path to a text file listing the sample IDs (file stems) to load.
        @param transform: The augmentation transformation to apply to images.
        """
        
        self.root = Path(root)
        self.transform = transform
        
        with open(self.root / split_txt) as f:
            self.stems = [line.strip() for line in f if line.strip()]
        
        self.img_dir = self.root / "images"
        self.mask_dir = self.root / "masks"

    def __len__(self):
        return len(self.stems)

    def __getitem__(self, idx):
        stem = self.stems[idx]
        img_p = str(self.img_dir / f"{stem}.jpg")
        mask_p = str(self.mask_dir / f"{stem}.png")

        img = cv2.cvtColor(cv2.imread(img_p, cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)
        mask = cv2.imread(mask_p, cv2.IMREAD_GRAYSCALE)
        mask = (mask > 0).astype(np.uint8) # binary

        # Albumentations needs dict, it will resize/pad consistently
        augmented = self.transform(image=img, mask=mask)
        img_t = augmented["image"] # FloatTensor [C, H, W], normalized
        mask_t = augmented["mask"].unsqueeze(0).float() # [1, H, W] in {0, 1}
        
        return img_t, mask_t, stem

class UC3MLPDataset(Dataset):    
    def __init__(
        self,
        root: str,
        split: str = "train",
        split_txt: str | None = None,
        transform=None,
        mask_mode: str = "polygon",
        shrink_pct: float = 0.0,
        choose_lp: str = "largest"
    ):
        """
        UC3M-LP dataset loader for plate segmentation.

        Assumed directory structure:
        
        root/
            train.txt
            test.txt
            train/
                <stem>.jpg
                <stem>.json
            test/
                <stem>.jpg
                <stem>.json

        @param root: Dataset root directory.
        @param split: "train" or "test".
        @param split_txt: Override, path to a text file listing the sample IDs (file stems) to load.
        @param transform: The augmentation transformation to apply to images.
        @param mask_mode: "polygon" (recommended) or "bbox".
        @param shrink_pct: Optional shrink of polygon/box.
        @param choose_lp: If multiple plates on image: "largest" or "first".
        """

        self.root = Path(root)
        assert split in ("train", "test"), "split must be 'train' or 'test'"
        self.split = split
        self.transform = transform
        self.mask_mode = mask_mode
        self.shrink_pct = float(shrink_pct)
        self.choose_lp = choose_lp

        # Determine which stem list to use
        if split_txt is None:
            split_txt = f"{split}.txt"  # root/train.txt or root/test.txt

        stems_path = self.root / split_txt
        with open(stems_path, "r", encoding="utf-8") as f:
            self.stems = [ln.strip() for ln in f if ln.strip()]

        self.data_dir = self.root / split  # root/train or root/test

    def __len__(self):
        return len(self.stems)

    def __getitem__(self, idx):
        stem = self.stems[idx]

        img_path = self.data_dir / f"{stem}.jpg"
        ann_path = self.data_dir / f"{stem}.json"

        # Read image (RGB)
        img_bgr = cv2.imread(str(img_path), cv2.IMREAD_COLOR)
        if img_bgr is None:
            raise FileNotFoundError(f"Could not read image: {img_path}")
        img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

        # Read annotation JSON
        with open(ann_path, "r", encoding="utf-8") as f:
            ann = json.load(f)

        h = int(ann.get("imageHeight", img.shape[0]))
        w = int(ann.get("imageWidth", img.shape[1]))

        # Build binary mask from lps[*]
        mask = np.zeros((h, w), dtype=np.uint8)
        lps = ann.get("lps", [])

        if len(lps) == 0:
            # No plate: keep empty mask
            pass
        else:
            lp = self._select_lp(lps)
            if self.mask_mode == "polygon":
                poly = np.array(lp["poly_coord"], dtype=np.float32)  # [[x,y],...]
                poly = self._maybe_shrink_polygon(poly)
                poly_i = np.round(poly).astype(np.int32)
                cv2.fillPoly(mask, [poly_i], 1)
            elif self.mask_mode == "bbox":
                # Derive bbox from polygon if desired
                poly = np.array(lp["poly_coord"], dtype=np.float32)
                x1, y1 = poly.min(axis=0)
                x2, y2 = poly.max(axis=0)
                x1, y1, x2, y2 = self._maybe_shrink_box(x1, y1, x2, y2)
                x1, y1, x2, y2 = map(int, map(round, (x1, y1, x2, y2)))
                x1, y1 = max(0, x1), max(0, y1)
                x2, y2 = min(w, x2), min(h, y2)
                if x2 > x1 and y2 > y1:
                    mask[y1:y2, x1:x2] = 1
            else:
                raise ValueError("mask_mode must be 'polygon' or 'bbox'")

        # Apply transform consistently (image + mask)
        if self.transform is not None:
            out = self.transform(image=img, mask=mask)
            img_out = out["image"]
            mask_out = out["mask"]
            # Ensure [1,H,W] float mask for training
            if isinstance(mask_out, torch.Tensor):
                mask_out = mask_out.unsqueeze(0).float()
            else:
                mask_out = mask_out[None, ...].astype(np.float32)
            return img_out, mask_out, stem

        # No transform: return numpy arrays
        return img, mask[None, ...].astype(np.float32), stem

    def _select_lp(self, lps):
        if self.choose_lp == "first":
            return lps[0]
        
        # largest polygon area
        def poly_area(lp):
            pts = np.array(lp["poly_coord"], dtype=np.float32)
            return float(cv2.contourArea(pts))
        
        return max(lps, key=poly_area)

    def _maybe_shrink_polygon(self, poly_xy: np.ndarray) -> np.ndarray:
        """
        Shrink polygon toward its centroid by shrink_pct to reduce background.
        poly_xy: [N,2] float32
        """
        
        if self.shrink_pct <= 0:
            return poly_xy
        
        c = poly_xy.mean(axis=0, keepdims=True)
        
        return c + (1.0 - self.shrink_pct) * (poly_xy - c)

    def _maybe_shrink_box(self, x1, y1, x2, y2):
        if self.shrink_pct <= 0:
            return x1, y1, x2, y2
        
        cx, cy = (x1 + x2) / 2.0, (y1 + y2) / 2.0
        w, h = (x2 - x1), (y2 - y1)
        w2, h2 = (1.0 - self.shrink_pct) * w / 2.0, (1.0 - self.shrink_pct) * h / 2.0
        
        return cx - w2, cy - h2, cx + w2, cy + h2

def extract_plate_crop(img_bgr, mask_bin, out_h=32, max_w=256, use_adaptive=False):
    """
    OCR crop normalization helper.
    
    @param img_bgr: Original BGR image (uint8).
    @param mask_bin: Binary mask (uint8 {0, 1}) same height and width.
    @param out_h: Output image height.
    @param max_w: Maximum width of output image.
    @param use_adaptive: Use adaptive threshold.
    @return: A normalized plate crop for OCR.
    """
    
    mask = (mask_bin > 0).astype(np.uint8)
    
    if mask.sum() == 0:
        return None # no plate found

    # largest connected component
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    
    if num_labels <= 1:
        return None
    
    largest = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
    cc = (labels == largest).astype(np.uint8)

    # get contour and minAreaRect for rough rectification
    contours, _ = cv2.findContours(cc, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    max_area_contour = max(contours, key=cv2.contourArea)
    contour_rect = cv2.minAreaRect(max_area_contour)
    contour_box = cv2.boxPoints(contour_rect).astype(np.float32)

    # order box points (tl, tr, br, bl)
    s = contour_box.sum(axis=1)
    diff = np.diff(contour_box, axis=1).reshape(-1)
    top_left     = contour_box[np.argmin(s)]
    bottom_right = contour_box[np.argmax(s)]
    top_right    = contour_box[np.argmin(diff)]
    bottom_left  = contour_box[np.argmax(diff)]
    
    contour_box = np.array([top_left, top_right, bottom_right, bottom_left], dtype=np.float32)

    # width/height for warp (keep aspect roughly plate-like)
    width  = int(max(np.linalg.norm(contour_box[1] - contour_box[0]), np.linalg.norm(contour_box[2] - contour_box[3])))
    height = int(max(np.linalg.norm(contour_box[3] - contour_box[0]), np.linalg.norm(contour_box[2] - contour_box[1])))
    width  = max(width, 1)
    height = max(height, 1)

    dst = np.array([[0, 0], [width - 1, 0], [width - 1, height - 1], [0, height - 1]], dtype=np.float32)
    M = cv2.getPerspectiveTransform(contour_box, dst)
    warp = cv2.warpPerspective(img_bgr, M, (width, height), flags=cv2.INTER_CUBIC)

    # grayscale + contrast normalization
    gray = cv2.cvtColor(warp, cv2.COLOR_BGR2GRAY)

    # CLAHE helps a lot on low-light/overexposed plates
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    norm = clahe.apply(gray)

    # resize to OCR-friendly height
    scale = out_h / norm.shape[0]
    new_w = min(int(round(norm.shape[1] * scale)), max_w)
    ocr_img = cv2.resize(norm, (new_w, out_h), interpolation=cv2.INTER_CUBIC)

    if use_adaptive:
        ocr_img = cv2.adaptiveThreshold(
            ocr_img, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
            cv2.THRESH_BINARY, blockSize=11, C=2
        )
        
    return ocr_img # uint8, H = out_h, W <= max_w

## U-Net Building Blocks

In [ ]:
class ConvBlock(nn.Module):
    """(Conv -> BN -> ReLU) x 2"""
    
    def __init__(self, in_ch: int, out_ch: int) -> None:
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)

class Down(nn.Module):
    """Downscale with maxpool then double conv"""
    
    def __init__(self, in_ch: int, out_ch: int) -> None:
        super().__init__()
        self.pool = nn.MaxPool2d(2)
        self.conv = ConvBlock(in_ch, out_ch)

    def forward(self, x):
        x = self.pool(x)
        return self.conv(x)

class Up(nn.Module):
    """Upscale then double conv. Uses transposed conv for upsampling."""
    
    def __init__(self, in_ch: int, out_ch: int) -> None:
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, in_ch // 2, kernel_size=2, stride=2)
        self.conv = ConvBlock(in_ch, out_ch) # in_ch = skip(ch) + up(ch)

    def forward(self, x, skip):
        x = self.up(x)
        
        # pad if needed (handles odd dims)
        diff_y = skip.size(-2) - x.size(-2)
        diff_x = skip.size(-1) - x.size(-1)
        
        x = F.pad(x, [diff_x // 2, diff_x - diff_x // 2,
                      diff_y // 2, diff_y - diff_y // 2])
        x = torch.cat([skip, x], dim=1)
        
        return self.conv(x)

class UNetSmall(nn.Module):
    """
    Lightweight U-Net for binary segmentation (1 class: plate vs background).
    Input: 3xHxW, Output: 1xHxW logits (use with BCEWithLogits).
    """
    
    def __init__(self, in_ch: int = 3, base_ch: int = 32) -> None:
        super().__init__()
        
        # Encoder
        self.inc  = ConvBlock(in_ch, base_ch)       # 3 -> 32
        self.down1 = Down(base_ch, base_ch * 2)     # 32 -> 64
        self.down2 = Down(base_ch * 2, base_ch * 4) # 64 -> 128
        self.down3 = Down(base_ch * 4, base_ch * 8) # 128 -> 256
        
        # Optional extra depth for larger model
        # self.down4 = Down(base_ch * 8, base_ch * 16)

        # Bottleneck
        self.bottleneck = ConvBlock(base_ch * 8, base_ch * 16)

        # Decoder
        self.up3 = Up(base_ch * 16, base_ch * 8) # (256 + 256) -> 256
        self.up2 = Up(base_ch * 8,  base_ch * 4) # (128 + 128) -> 128
        self.up1 = Up(base_ch * 4,  base_ch * 2) # (64 + 64)   -> 64
        self.up0 = Up(base_ch * 2,  base_ch)     # (32 + 32)   -> 32

        # Head
        self.outc = nn.Conv2d(base_ch, 1, kernel_size=1)

    def forward(self, x):
        x0 = self.inc(x)    # H
        x1 = self.down1(x0) # H/2
        x2 = self.down2(x1) # H/4
        x3 = self.down3(x2) # H/8
        
        xb = self.bottleneck(x3)

        y3 = self.up3(xb, x3)
        y2 = self.up2(y3, x2)
        y1 = self.up1(y2, x1)
        y0 = self.up0(y1, x0)
        
        logits = self.outc(y0) # raw logits
        
        return logits

## Loss & Simple Metrics (Dice + BCE)

In [ ]:
class DiceLoss(nn.Module):
    """Soft Dice loss for binary segmentation."""
    
    def __init__(self, eps: float = 1e-6) -> None:
        super().__init__()
        self.eps = eps

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        # logits: [B, 1, H, W], targets: [B, 1, H, W] in {0, 1}
        probs = torch.sigmoid(logits)
        num = 2.0 * (probs * targets).sum(dim=(2, 3)) + self.eps
        den = (probs.pow(2) + targets.pow(2)).sum(dim=(2, 3)) + self.eps
        dice = 1.0 - (num / den)
        return dice.mean()

def bce_dice_loss(logits: torch.Tensor, targets: torch.Tensor, bce_weight: float = 0.5) -> torch.Tensor:
    bce = F.binary_cross_entropy_with_logits(logits, targets)
    dice = DiceLoss()(logits, targets)
    return bce_weight * bce + (1 - bce_weight) * dice

@torch.no_grad()
def iou_score(logits: torch.Tensor,
    targets: torch.Tensor,
    thr: float = 0.5,
    eps: float = 1e-6
) -> float:
    probs = torch.sigmoid(logits)
    preds = (probs > thr).float()
    inter = (preds * targets).sum(dim=(2, 3))
    union = (preds + targets - preds * targets).sum(dim=(2, 3)) + eps
    iou = (inter + eps) / union
    return iou.mean().item()

## Training Functions

In [ ]:
def create_model(
    device: str | torch.device ="cuda" if torch.cuda.is_available() else "cpu"
) -> tuple[torch.nn.Module, torch.optim.Optimizer, torch.device]:
    model = UNetSmall(in_ch=3, base_ch=32).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
    return model, optimizer, device

def load_model(
    model: torch.nn.Module,
    checkpoint_path: str | Path,
    device: str | torch.device | None = None,
    strict: bool = True
) -> torch.nn.Module:
    """
    Load model weights from a checkpoint file.

    @param model: Instantiated model (architecture must match checkpoint).
    @param checkpoint_path: Path to .pt or .pth file.
    @param device: Target device ("cpu", "cuda", torch.device). If None, auto-detect.
    @param strict: Whether to strictly enforce that the keys in state_dict match.

    @return: Model with loaded weights, moved to the target device.
    """
    checkpoint_path = Path(checkpoint_path)
    assert checkpoint_path.is_file(), f"Checkpoint not found: {checkpoint_path}"

    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    else:
        device = torch.device(device)

    # Load checkpoint to CPU first (safe and portable)
    checkpoint = torch.load(checkpoint_path, map_location="cpu")

    # Support both raw state_dict and wrapped checkpoints
    if isinstance(checkpoint, dict) and "state_dict" in checkpoint:
        state_dict = checkpoint["state_dict"]
    else:
        state_dict = checkpoint

    # Handle DataParallel / DDP prefixes if present
    if any(k.startswith("module.") for k in state_dict.keys()):
        state_dict = {k.replace("module.", "", 1): v for k, v in state_dict.items()}

    model.load_state_dict(state_dict, strict=strict)
    model.to(device)
    model.eval()

    return model

def train_one_epoch(
    model: torch.nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
) -> float:
    model.train()
    total_loss = 0.0
    
    for imgs, masks, _ in loader:
        imgs, masks = imgs.to(device), masks.to(device)
        logits = model(imgs)
        loss = bce_dice_loss(logits, masks, bce_weight=0.5)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
    
    return total_loss / len(loader.dataset)

@torch.no_grad()
def validate(
    model: torch.nn.Module,
    loader: DataLoader,
    device: torch.device,
) -> tuple[float, float]:
    model.eval()
    total_loss, total_iou, n = 0.0, 0.0, 0
    
    for imgs, masks, _ in loader:
        imgs, masks = imgs.to(device), masks.to(device)
        logits = model(imgs)
        loss = bce_dice_loss(logits, masks, bce_weight=0.5)
        total_loss += loss.item() * imgs.size(0)
        total_iou += iou_score(logits, masks) * imgs.size(0)
        n += imgs.size(0)
        
    return total_loss / n, total_iou / n

## Training Loop

In [ ]:
train_ds = UC3MLPDataset(
    root="datasets/UC3M-LP",
    split="train",
    transform=get_train_transform(target_hw=(512, 512)),
    mask_mode="polygon",
    shrink_pct=0.05,
    choose_lp="largest"
)

test_ds = UC3MLPDataset(
    root="datasets/UC3M-LP",
    split="test",
    transform=get_train_transform(target_hw=(512, 512)),
    mask_mode="polygon",
    shrink_pct=0.05,
    choose_lp="largest"
)

train_dl = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=0, pin_memory=True)
test_dl = DataLoader(test_ds, batch_size=8, shuffle=False, num_workers=4, pin_memory=True)

model, optim, device = create_model()

for epoch in range(20):
    train_loss = train_one_epoch(model, train_dl, optim, device)
    test_loss, test_iou = validate(model, test_dl, device)
    print(f"Epoch {epoch:02d} | train {train_loss:.4f} | test {test_loss:.4f} | IoU {test_iou:.4f}")

MODEL_PATH = f"./model-{int(time.time())}.pt"

torch.save(model.state_dict(), MODEL_PATH)